In [0]:
# Gold layer: fact_leads (Kimball transaction-grain fact - marketing leads)
# Source: b2b_ecommerce.silver.leads - the marketing-spreadsheet stream,
#         one row per lead_id (upsert, latest wins - silver owns validation).
# Grain:  exactly one row per lead_id (the lead grain) - the MERGE key,
#         so retries and re-runs are idempotent.
# Joins:  date_sk = dim_date's smart YYYYMMDD key from created_date (all
#         1,098 dates valid: 0 nulls, 0 outside the dim_date range that
#         was built including leads.created_date - no validity flag
#         needed, unlike fact_order_items' order_date). company_sk is
#         the conversion bridge for KPI 14 (lead funnel) / KPI 15 (lead
#         score vs order value): leads carry NO company_id, so a lead is
#         linked to its buyer company by an exact case-insensitive
#         company_name match against the CURRENT dim_companies version
#         (SCD2 __END_AT IS NULL - a Won lead becomes today's company).
#         Pre-flight (2026-11-19): 144 of 1,098 leads match (all 144 are
#         Won - only won leads name an existing customer company), each
#         against exactly ONE company (0 multi-match), and those 144
#         companies hold 3,530 orders in silver - the bridge carries
#         real conversion volume. Unmatched leads (New/Contacted/
#         Qualified/Proposal/Lost + 17 Won not yet in dim_companies)
#         keep their row with NULL company_sk - facts retain rows, they
#         never drop them. KPI views aggregate funnel stages from
#         status and correlate via company_sk -> fact_order_items.
# Columns: lead_id (business + MERGE key), date_sk, raw created_date,
#         company_sk (conversion bridge), source + campaign_name +
#         utm_source/utm_medium/utm_campaign (the KPI 14 campaign /
#         channel attribution set - kept whole: pre-flight showed the
#         UTMs are NOT redundant with source/campaign_name, e.g.
#         utm_campaign differs from campaign_name in all 1,098 rows),
#         status (funnel stage: New/Contacted/Qualified/Proposal/Won/
#         Lost), lead_score + estimated_order_volume (the KPI 15 lead
#         quality measures; 82 / 26 nulls - retained, never dropped).
#         Excluded per the KPI audit (no KPI uses them; one-line
#         additions if reports later need them): country, city,
#         company_name (raw bridge input - company_sk supersedes it; the
#         dim owns names), company_size, industry, contact_name,
#         contact_email, phone, job_title, sales_rep, last_contacted_date,
#         utm_content, source_file, updated_at (gold timestamp policy:
#         silver updated_at mixes ART/UTC clocks - gold excludes it).
# Storage: Delta-backed STREAMING TABLE (facts are stored tables, not
#         views), loaded via AUTO CDC FROM SNAPSHOT - the same
#         ingestion-time pattern as the gold dims and fact_order_items.
# MERGE / idempotency: keyed on lead_id (requirements' MERGE
#         constraint - safe to retry without duplicates). The snapshot
#         view is recomputed each run; an unchanged source diffs to zero
#         events and changes nothing. Type 1 - facts do not version
#         history; the SCD2 dim owns that.
# Quality: no expectations - AUTO CDC FROM SNAPSHOT flows cannot carry
#         them (same decision as the dims and fact_order_items), and
#         silver already owns row-level validation.
# Layout: partitioned by date_sk (requirements: fact_* partitioned by
#         date). Delta does not allow clustering keys alongside
#         partition columns
#         (SPECIFY_CLUSTER_BY_WITH_PARTITIONED_BY_IS_NOT_ALLOWED), so no
#         cluster_by - same decision as fact_order_items; the date
#         partitions already prune MERGE scans and the table is tiny
#         (~1.1K rows).
from pyspark import pipelines as dp
from pyspark.sql import functions as F

GOLD = "b2b_ecommerce.gold"
SILVER_LEADS = "b2b_ecommerce.silver.leads"


# The complete fact snapshot, recomputed at each pipeline run: the
# silver leads stream, company_sk-resolved via the name bridge, trimmed
# to the KPI-required columns. A batch temporary view - the engine
# reads it at each run (ingestion-time snapshot versioning). No
# driver-side actions.
@dp.temporary_view(name="fact_leads_snapshot")
def fact_leads_snapshot():
    # The conversion bridge: normalized company_name -> the CURRENT
    # (SCD2 open) dim_companies row. Exactly one company per matched
    # lead (pre-flight-verified 0 multi-match); unmatched leads pass
    # through with NULL company_sk (LEFT join - rows retained).
    current_companies = (
        spark.read.table(f"{GOLD}.dim_companies")
        .where(F.col("__END_AT").isNull())
        .select(
            F.lower(F.trim(F.col("name"))).alias("_company_name_norm"),
            "company_sk",
        )
    )

    return (
        spark.read.table(SILVER_LEADS)
        .join(
            current_companies,
            F.lower(F.trim(F.col("company_name")))
            == F.col("_company_name_norm"),
            "left",
        )
        .withColumn(
            "date_sk",
            (
                F.year("created_date") * 10000
                + F.month("created_date") * 100
                + F.dayofmonth("created_date")
            ).cast("int"),
        )
        .select(
            "lead_id",
            "date_sk",
            "created_date",
            "company_sk",
            "source",
            "campaign_name",
            "utm_source",
            "utm_medium",
            "utm_campaign",
            "status",
            "lead_score",
            "estimated_order_volume",
        )
    )


# Delta-backed streaming table: the AUTO CDC target. Partitioned by the
# date key (requirements: fact_* partitioned by date). No clustering
# keys: Delta forbids CLUSTER BY together with PARTITIONED BY.
dp.create_streaming_table(
    name="fact_leads",
    comment=(
        "Kimball transaction-grain fact: one row per lead_id from the silver "
        "leads stream. date_sk = YYYYMMDD from created_date (all dates valid). "
        "company_sk bridges a lead to its buyer company via an exact "
        "case-insensitive company_name match against the CURRENT dim_companies "
        "version - leads carry no company_id, so this name match is the only "
        "conversion path for the lead-funnel and lead-quality KPIs (144 of "
        "1,098 leads match, all Won, each uniquely; unmatched leads are "
        "retained with NULL company_sk). status is the funnel stage; source / "
        "campaign_name / utm_source / utm_medium / utm_campaign carry the "
        "campaign-channel attribution; lead_score + estimated_order_volume "
        "are the quality measures. MERGE-loaded via AUTO CDC FROM SNAPSHOT "
        "keyed on lead_id (idempotent retries). Partitioned by date_sk "
        "(requirements: fact_* partitioned by date)."
    ),
    partition_cols=["date_sk"],
)


# The auto-merge load: diff each run's fact snapshot against the last
# committed one and apply inserts/updates/deletes in place (Type 1 -
# facts do not version history; the SCD2 dims own that).
dp.create_auto_cdc_from_snapshot_flow(
    target="fact_leads",
    source="fact_leads_snapshot",  # ingestion-time snapshot versioning
    keys=["lead_id"],
    stored_as_scd_type=1,
)